In [ ]:
import os
for dirname, _, filenames in os.walk('/GitHub/Fake News Real News/News'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os
import re
import string
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

FAKE_PATH = "/Users/faizd/GitHub/Fake News Real News/News/Fake.csv"
TRUE_PATH  = "/Users/faizd/GitHub/Fake News Real News/News/True.csv"
OUT_DIR    = "/Users/faizd/GitHub/Fake News Real News/News/"

os.makedirs(OUT_DIR, exist_ok=True)

print("📂  Loading datasets...")
data_fake = pd.read_csv(FAKE_PATH)
data_true  = pd.read_csv(TRUE_PATH)

data_fake["class"] = 0
data_true["class"]  = 1

data_fake = data_fake.iloc[:-10]
data_true  = data_true.iloc[:-10]

data = pd.concat([data_fake, data_true], axis=0)
data = data.drop(["title", "subject", "date"], axis=1)
data = data.dropna(subset=["text"])
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅  Dataset: {len(data):,} rows  |  Fake: {(data['class']==0).sum():,}  |  Real: {(data['class']==1).sum():,}")

def wordopt(text: str) -> str:
    text = text.lower()
    text = re.sub(r'^[a-z\s,\.]+\([^)]+\)\s*[-–—]\s*', '', text)
    text = re.sub(r'\bby [a-z]+ [a-z]+\b', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\b(?=[a-z]+\d|\d+[a-z])[a-z0-9]+\b', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("🔧  Preprocessing text...")
data["text"] = data["text"].apply(wordopt)

x = data["text"]
y = data["class"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

print("📐  Fitting TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=50000,
    min_df=5,
    max_df=0.85,
    sublinear_tf=True,
    ngram_range=(1, 2),
)

xv_train = vectorizer.fit_transform(x_train)
xv_test  = vectorizer.transform(x_test)

joblib.dump(vectorizer, os.path.join(OUT_DIR, "vectorizer.pkl"))
print(f"    Saved → {OUT_DIR}/vectorizer.pkl")
print(f"    Vocabulary size: {len(vectorizer.vocabulary_):,} features")

classifiers = {
    "lr": LogisticRegression(C=1.0, max_iter=1000, solver='saga', random_state=42),
    "dt": DecisionTreeClassifier(max_depth=20, min_samples_leaf=5, random_state=42),
    "gb": GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=0),
    "rf": RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=5, max_features='sqrt', random_state=0, n_jobs=-1),
}

labels = {
    "lr": "Logistic Regression",
    "dt": "Decision Tree",
    "gb": "Gradient Boost",
    "rf": "Random Forest",
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for key, clf in classifiers.items():
    print(f"\n🚀  Training {labels[key]}...")
    clf.fit(xv_train, y_train)

    preds    = clf.predict(xv_test)
    accuracy = accuracy_score(y_test, preds)
    print(f"    Hold-out Accuracy: {accuracy * 100:.2f}%")
    print(classification_report(y_test, preds, target_names=["Fake", "Real"]))

    print(f"    Running 5-fold CV...")
    cv_scores = cross_val_score(clf, xv_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    print(f"    CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    if accuracy - cv_scores.mean() > 0.05:
        print(f"    ⚠️  Warning: hold-out acc is much higher than CV F1 — possible overfitting")

    out_path = os.path.join(OUT_DIR, f"{key}.pkl")
    joblib.dump(clf, out_path)
    print(f"    Saved → {out_path}")

print("\n" + "="*60)
print("🧪  OUT-OF-DISTRIBUTION TEST")
print("="*60)

OOD_SAMPLES = [
    ("The Federal Reserve raised interest rates by 25 basis points on Wednesday, as policymakers continued efforts to curb inflation, officials confirmed.", 1),
    ("Parliament approved a new climate bill on Thursday after months of debate. The legislation sets binding emissions targets for 2035, according to officials.", 1),
    ("Apple Inc reported quarterly revenue of 89.5 billion on Thursday, topping Wall Street estimates as strong iPhone sales offset a slowdown in services. Chief Executive Tim Cook said demand for the latest models remained robust.", 1),
    ("Microsoft Corp posted net income of 22.3 billion in its fiscal first quarter, beating analyst expectations as its Azure cloud computing unit grew rapidly. Chief Executive Satya Nadella said artificial intelligence features were driving customer adoption. Shares climbed 3.8 percent in extended trading.", 1),
    ("Tesla Inc delivered 435,059 vehicles in the third quarter, falling short of Wall Street expectations of 455,000 units as supply chain disruptions weighed on production. Chief Executive Elon Musk said planned factory shutdowns partly caused the miss. Shares fell 4.5 percent in premarket trading.", 1),
    ("SHOCKING: Scientists ADMIT the moon landing was staged!! Share before NASA deletes this! They don't want you to know the truth.", 0),
    ("Big pharma is hiding a miracle cure from you. A whistleblower exposed that a simple herb completely destroys cancer cells. Wake up and share the truth!", 0),
    ("The deep state is planning to microchip all citizens by 2025. Mainstream media refuses to cover this. They are hiding this from the public.", 0),
    ("A leaked government document confirms that fluoride in water is intentionally used to lower IQ and make citizens easier to control. Scientists are silenced.", 0),
]

for key, clf in classifiers.items():
    print(f"\n  [{labels[key]}]")
    correct = 0
    for text, label in OOD_SAMPLES:
        cleaned = wordopt(text)
        vec     = vectorizer.transform([cleaned])
        pred    = clf.predict(vec)[0]
        status  = "✅" if pred == label else "❌"
        tag     = "Real" if label == 1 else "Fake"
        if pred == label:
            correct += 1
        print(f"    {status} Expected={tag:4s}  Got={'Real' if pred==1 else 'Fake':4s}  | {text[:65]}...")
    print(f"    OOD accuracy: {correct}/{len(OOD_SAMPLES)} = {correct/len(OOD_SAMPLES)*100:.0f}%")

📂  Loading datasets...
✅  Dataset: 44,878 rows  |  Fake: 23,471  |  Real: 21,407
🔧  Preprocessing text...
📐  Fitting TF-IDF vectorizer...
    Saved → /Users/faizd/Desktop/News//vectorizer.pkl
    Vocabulary size: 50,000 features

🚀  Training Logistic Regression...
    Hold-out Accuracy: 98.74%
              precision    recall  f1-score   support

        Fake       0.99      0.99      0.99      5868
        Real       0.99      0.99      0.99      5352

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220

    Running 5-fold CV...
    CV F1: 0.9843 ± 0.0009
    Saved → /Users/faizd/Desktop/News/lr.pkl

🚀  Training Decision Tree...
    Hold-out Accuracy: 95.38%
              precision    recall  f1-score   support

        Fake       0.95      0.96      0.96      5868
        Real       0.96      0.95      0.95      5352

    accuracy                           0.95     11220
   